In [ ]:
## %pip install qiskit-aer qiskit-algorithms qiskit-ibm-runtime qiskit-nature pyscf networkx

# VQE for LiH

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import qiskit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

# Qiskit Nature (electronic structure -> qubit Hamiltonian)
import qiskit_nature
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

# PySCF (classical electronic structure backend)
import pyscf


In [ ]:
## build the LiH electronic structure problem at equilibrium

# Equilibrium bond length (commonly used 
bond_length = 1.5474  # Angstrom

driver = PySCFDriver(
    atom=f"Li 0.0 0.0 0.0; H 0.0 0.0 {bond_length}",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

problem = driver.run()

print(f"Number of spatial orbitals : {problem.num_spatial_orbitals}")
print(f"Number of spin orbitals    : {2 * problem.num_spatial_orbitals}")
print(f"Number of particles        : {problem.num_particles}")
print(f"Nuclear repulsion energy   : {problem.nuclear_repulsion_energy:.6f} Hartree")
print(f"Reference HF energy        : {problem.reference_energy:.6f} Hartree")

In [ ]:
"""
To reproduce Kandala et al. (Nature 549, 242-246, 2017): (not 100%)
- Active orbitals: H 1s (MO1), Li 2s (MO2), Li 2px (MO3) → 3 spatial orbitals
- Active electrons: 4 (including frozen Li 1s contribution) → particles = (2,2)
- Frozen core energy shift (-7.818690 Hartree) is extracted from
  ActiveSpaceTransformer(2e, 3 orbitals) and manually added back,
  since ActiveSpaceTransformer(4e, 3 orbitals) does not compute it.
- Result: 4 qubits, 100 Pauli terms (paper: 99, difference may be IIII constant term handling)
"""

# using (2e, 3 orbitals) get correct energy shift
problem_ref = driver.run()
ref_transformer = ActiveSpaceTransformer(num_electrons=2, num_spatial_orbitals=3)
problem_ref = ref_transformer.transform(problem_ref)
frozen_core_shift = problem_ref.hamiltonian.constants["ActiveSpaceTransformer"]

# then using (4e, 3 orbitals) to construct Hamiltonian
problem = driver.run()
active_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)
problem = active_transformer.transform(problem)
problem.hamiltonian.constants["frozen_core"] = frozen_core_shift

print(f"Hamiltonian constants : {problem.hamiltonian.constants}")

In [ ]:
## map fermionic Hamiltonian to qubit Hamiltonian
mapper = ParityMapper(num_particles=problem.num_particles)
qubit_hamiltonian = mapper.map(problem.hamiltonian.second_q_op())

print(f"Number of qubits      : {qubit_hamiltonian.num_qubits}")
print(f"Number of Pauli terms : {len(qubit_hamiltonian)}")

print(f"\nFirst 5 Pauli terms:")
for i, (pauli, coeff) in enumerate(qubit_hamiltonian.to_list()):
    if i >= 5: break
    print(f"  {pauli} : {coeff.real:+.6f}")

# FCI reference -> diagonalize the 4-qubit Hamiltonian directly
H_matrix = qubit_hamiltonian.to_matrix()
fci_electronic = float(np.linalg.eigvalsh(H_matrix)[0])
print(f"\nFCI electronic energy : {fci_electronic:.6f} Hartree")

fci_total = fci_electronic + problem.nuclear_repulsion_energy
print(f"FCI total energy      : {fci_total:.6f} Hartree")
print(f"Expected (paper)      : ~-7.78 Hartree @ R = 1.5474 Å")

In [ ]:
# # UCCSD ansatz with Hartree-Fock initial state

# # Hartree-Fock state --> the classical mean-field reference
# hf_state = HartreeFock(
#     num_spatial_orbitals=problem.num_spatial_orbitals,
#     num_particles=problem.num_particles,
#     qubit_mapper=mapper,
# )

# # UCCSD ansatz 
# ansatz = UCCSD(
#     num_spatial_orbitals=problem.num_spatial_orbitals,
#     num_particles=problem.num_particles,
#     qubit_mapper=mapper,
#     initial_state=hf_state,
# )

# print(f"Number of qubits      : {ansatz.num_qubits}")
# print(f"Number of parameters  : {ansatz.num_parameters}")
# print(f"Circuit depth         : {ansatz.decompose().depth()}")

In [ ]:
## HEA ansatz tailored to cross 5q topology (center = q2)
# entanglement follows physical connectivity: q0-q2, q1-q2, q3-q2
from qiskit.circuit.library import efficient_su2


ansatz = efficient_su2(
    num_qubits=4,
    reps=1,
    entanglement=[[0,2],[1,2],[3,2]],
    su2_gates=["rz", "ry"]          # change here for native single qubit gate  (in paper ZXZ)
)   

print(f"Number of qubits     : {ansatz.num_qubits}")
print(f"Number of parameters : {ansatz.num_parameters}")  
print(f"Circuit depth        : {ansatz.decompose().depth()}")

In [ ]:
## Noiseless VQE benchmark using HEA ansatz with **L-BFGS-B** optimizer to verify ansatz expressibility against FCI
estimator = StatevectorEstimator()

history = {"iter": 0, "energies": []}

def cost_function(params):
    """Compute <psi(params) | H | psi(params)>"""
    job = estimator.run([(ansatz, qubit_hamiltonian, [params])])
    energy = float(np.asarray(job.result()[0].data.evs).reshape(-1)[0])
    history["iter"] += 1
    history["energies"].append(energy)
    return energy

# energy at theta = 0 (HEA initial state, not necessarily HF)
# x0 = np.zeros(ansatz.num_parameters)
# initial_energy = cost_function(x0)

rng = np.random.default_rng(42)
x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)
initial_energy = cost_function(x0)
print(f"Initial energy (theta=0) : {initial_energy:.6f} Hartree")

history = {"iter": 0, "energies": []}

print("\nRunning VQE (HEA + L-BFGS-B, noiseless)...")
result = minimize(
    cost_function,
    x0=x0,
    method="L-BFGS-B",
    options={"maxiter": 200, "ftol": 1e-12, "gtol": 1e-8},
)

vqe_energy_electronic = float(result.fun)
n_evals = history["iter"]

error_mHa = (vqe_energy_electronic - fci_electronic) * 1000
print(f"\nVQE electronic energy : {vqe_energy_electronic:.6f} Hartree")
print(f"FCI electronic energy : {fci_electronic:.6f} Hartree")
print(f"Error                 : {error_mHa:+.4f} mHartree")
print(f"Chemical accuracy     : {'YES' if abs(error_mHa) < 1.6 else 'NO'} (threshold 1.6 mHa)")
print(f"Function evaluations  : {n_evals}")

plt.figure(figsize=(8, 4.5))
plt.plot(history["energies"], lw=1.5, marker="o", ms=4, label="HEA-VQE")
plt.axhline(fci_electronic, color="r", ls="--", lw=1, label=f"FCI = {fci_electronic:.4f}")
plt.axhline(initial_energy, color="gray", ls=":", lw=1, label=f"Initial = {initial_energy:.4f}")
plt.xlabel("Cost-function evaluations")
plt.ylabel("Electronic energy (Hartree)")
plt.title(f"VQE convergence — LiH @ R = {bond_length} Å (HEA, 4 qubits)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## Noiseless VQE benchmark using HEA ansatz with SPSA optimizer to validate optimizer behavior in noise-free conditions
from qiskit_algorithms.optimizers import SPSA 

"""
Noiseless should use L-BFGS-B
SPSA is for noisy env (since it only need 2 energy measurement)
So here, outcome is not good at all
"""

spsa = SPSA(
    maxiter=250,           
    blocking=False,        
    allowed_increase=0.1,  
    trust_region=False,    
    learning_rate=None,    
    perturbation=None,
    last_avg=25,            
    resamplings=1,        
    perturbation_dims=None, 
)

estimator = StatevectorEstimator()

history = {"iter": 0, "energies": []}

def cost_function(params):
    """Compute <psi(params) | H | psi(params)>"""
    job = estimator.run([(ansatz, qubit_hamiltonian, [params])])
    energy = float(np.asarray(job.result()[0].data.evs).reshape(-1)[0])
    history["iter"] += 1
    history["energies"].append(energy)
    return energy


print(f"Initial energy (theta=0) : {initial_energy:.6f} Hartree")

history = {"iter": 0, "energies": []}

print("\nRunning VQE (HEA + SPSA, noiseless)...")


result = spsa.minimize(cost_function, x0=x0)

vqe_energy_electronic = float(result.fun)
n_evals = history["iter"]

error_mHa = (vqe_energy_electronic - fci_electronic) * 1000
print(f"\nVQE electronic energy : {vqe_energy_electronic:.6f} Hartree")
print(f"FCI electronic energy : {fci_electronic:.6f} Hartree")
print(f"Error                 : {error_mHa:+.4f} mHartree")
print(f"Chemical accuracy     : {'YES' if abs(error_mHa) < 1.6 else 'NO'} (threshold 1.6 mHa)")
print(f"Function evaluations  : {n_evals}")

plt.figure(figsize=(8, 4.5))
plt.plot(history["energies"], lw=1.5, marker="o", ms=4, label="HEA-VQE (SPSA)")
plt.axhline(fci_electronic, color="r", ls="--", lw=1, label=f"FCI = {fci_electronic:.4f}")
plt.axhline(initial_energy, color="gray", ls=":", lw=1, label=f"Initial = {initial_energy:.4f}")
plt.xlabel("Cost-function evaluations")
plt.ylabel("Electronic energy (Hartree)")
plt.title(f"VQE convergence — LiH @ R = {bond_length} Å (HEA + SPSA, noiseless)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
## scan bond length and plot PEC (HEA, noiseless using L-BFGS-B and SPSA)

def run_vqe_at_distance(R, optimizer="lbfgsb"):
    """
    Full HEA-VQE pipeline at one Li-H distance.
    optimizer: "lbfgsb" or "spsa"
    Returns: (initial_total, VQE_total, FCI_total) in Hartree.
    """
    drv = PySCFDriver(
        atom=f"Li 0.0 0.0 0.0; H 0.0 0.0 {R}",
        basis="sto3g", charge=0, spin=0, unit=DistanceUnit.ANGSTROM,
    )

    prob_ref = drv.run()
    frozen_shift = ActiveSpaceTransformer(2, 3).transform(prob_ref).hamiltonian.constants["ActiveSpaceTransformer"]

    prob = drv.run()
    prob = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3).transform(prob)
    prob.hamiltonian.constants["frozen_core"] = frozen_shift

    mp = ParityMapper(num_particles=prob.num_particles)
    H = mp.map(prob.hamiltonian.second_q_op())
    shift = prob.nuclear_repulsion_energy
    fci_elec = float(np.linalg.eigvalsh(H.to_matrix())[0])

    ans = efficient_su2(
        num_qubits=4, reps=1,
        entanglement=[[0,2],[1,2],[3,2]],
        su2_gates=["rz", "ry"],
    )
    est = StatevectorEstimator()

    def cost(p):
        return float(np.asarray(est.run([(ans, H, [p])]).result()[0].data.evs).reshape(-1)[0])

    initial_elec = cost(np.zeros(ans.num_parameters))

    

    if optimizer == "lbfgsb":
        best_energy = np.inf
        for _ in range(3):
            x0 = rng.uniform(-np.pi, np.pi, ans.num_parameters)
            res = minimize(
                cost, x0=x0,
                method="L-BFGS-B",
                options={"maxiter": 200, "ftol": 1e-12, "gtol": 1e-8},
            )
            if res.fun < best_energy:
                best_energy = res.fun
    else:  # spsa 
        x0 = rng.uniform(-np.pi, np.pi, ans.num_parameters)
        res = spsa.minimize(cost, x0=x0)
        best_energy = float(res.fun)

    return initial_elec + shift, best_energy + shift, fci_elec + shift

distances = np.array([
    0.5, 0.6, 0.7, 0.8, 0.9,
    1.0, 1.1, 1.2, 1.3, 1.4,
    1.45, 1.50, 1.55, 1.60, 1.65, 1.70, 1.75, 1.80,
    2.0, 2.2, 2.4, 2.6, 2.8, 3.0,
    3.25, 3.5, 3.75, 4.0, 4.5, 5.0
])


E_initial, E_vqe_lbfgs, E_vqe_spsa, E_fci = [], [], [], []
for R in distances:
    initial, vqe_l, fci = run_vqe_at_distance(float(R), optimizer="lbfgsb")
    _,       vqe_s, _   = run_vqe_at_distance(float(R), optimizer="spsa")
    E_initial.append(initial)
    E_vqe_lbfgs.append(vqe_l)
    E_vqe_spsa.append(vqe_s)
    E_fci.append(fci)
    print(f"R={R:.2f}  L-BFGS-B={vqe_l:.4f}  SPSA={vqe_s:.4f}  FCI={fci:.4f}")

E_initial, E_vqe_lbfgs, E_vqe_spsa, E_fci = map(np.array, (E_initial, E_vqe_lbfgs, E_vqe_spsa, E_fci))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6.5), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1.2]})

ax1.plot(distances, E_vqe_lbfgs, "o--", color="#1f77b4", lw=1.6, ms=4, label="HEA-VQE (L-BFGS-B)")
ax1.plot(distances, E_vqe_spsa,  "s--", color="#d62728", lw=1.6, ms=4, label="HEA-VQE (SPSA)")
ax1.plot(distances, E_fci,       ":",   color="#2ca02c", lw=2.0, label="FCI")
ax1.set_ylabel("Energy (Hartree)")
ax1.legend(loc="upper right")
ax1.grid(alpha=0.3)
ax1.set_title("LiH dissociation curve — HEA noiseless, L-BFGS-B vs SPSA")

ax2.plot(distances, E_vqe_lbfgs - E_fci, "o--", color="#1f77b4", lw=1.4, ms=4, label="L-BFGS-B")
ax2.plot(distances, E_vqe_spsa  - E_fci, "s--", color="#d62728", lw=1.4, ms=4, label="SPSA")
ax2.axhline(0, color="k", lw=0.5)
ax2.axhline( 1.6e-3, color="gray", ls=":", lw=0.8)
ax2.axhline(-1.6e-3, color="gray", ls=":", lw=0.8)
ax2.text(distances[-1], 1.6e-3, "  chem. acc.", fontsize=8, color="gray", va="bottom")
ax2.set_xlabel("Li-H distance (Å)")
ax2.set_ylabel("Error vs FCI (Ha)")
ax2.legend(loc="upper left", fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
## logical circuit
ansatz.decompose().draw("mpl", style="clifford")


In [ ]:
## plot transpiled circuit of cross_5q topology 
from qiskit import transpile
from qiskit.transpiler import CouplingMap


# Cross 5-qubit coupling map (center = qubit 2) 
cross_5q = CouplingMap([
    [0,2],[2,0],
    [1,2],[2,1],
    [3,2],[2,3],
    [4,2],[2,4],
])

ansatz_transpiled = transpile(
    ansatz,
    coupling_map=cross_5q,
    basis_gates=["rz", "rx", "ry", "cz"],         # can change based on native gate of sc qubits
    optimization_level=3,                         # can set to 0, 1, 2, 3
    seed_transpiler=42,
    initial_layout={
        ansatz.qregs[0][0]: 1,
        ansatz.qregs[0][1]: 3,
        ansatz.qregs[0][2]: 2,
        ansatz.qregs[0][3]: 4,
    },
)

print(f"Depth  : {ansatz_transpiled.depth()}")
print(f"CZ   : {ansatz_transpiled.count_ops().get('cz', 0)}")
print(f"All ops: {ansatz_transpiled.count_ops()}")

ansatz_transpiled.draw("mpl", style="clifford")



# Noise from real backend(FakeFez)

### Choose topological structure on FakeBackend (ibm_fez)

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeFez
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit import transpile
from qiskit.transpiler import CouplingMap

fake_backend = FakeFez()

# extract noise for q2, q3, q4, q16 only (LiH uses these qubits)
noise_model_fez_full = NoiseModel.from_backend(fake_backend)
noise_model_fez = NoiseModel(basis_gates=noise_model_fez_full.basis_gates)

if hasattr(noise_model_fez_full, '_local_quantum_errors'):
    for gate, qubit_dict in noise_model_fez_full._local_quantum_errors.items():
        for qubits, error in qubit_dict.items():
            if all(q in [2, 3, 4, 16] for q in qubits):
                noise_model_fez.add_quantum_error(error, gate, list(qubits))

# only connections between q2, q3, q4, q16
coupling_map_fez = CouplingMap([
    [2,3],[3,2],
    [3,4],[4,3],
    [3,16],[16,3],
])
basis_gates_fez = noise_model_fez_full.basis_gates

initial_layout = {
    ansatz.qregs[0][0]: 2,
    ansatz.qregs[0][1]: 4,
    ansatz.qregs[0][2]: 3,   # center
    ansatz.qregs[0][3]: 16,
}

ansatz_fez = transpile(
    ansatz,
    backend=fake_backend,
    optimization_level=3,
    initial_layout=initial_layout,
    seed_transpiler=42,
)

ops = ansatz_fez.count_ops()
print(f"Depth  : {ansatz_fez.depth()}")
print(f"CZ     : {ops.get('cz', 0)}")
print(f"All ops: {ops}")
print(f"Layout : {ansatz_fez.layout.final_index_layout()}")

In [ ]:
## calculate fixed distance vqe of HEA with spsa using FakeFez backend
qubit_ham_fez = qubit_hamiltonian.apply_layout(ansatz_fez.layout)

noisy_estimator_fez = AerEstimator(
    options={
        "backend_options": {
            "noise_model": noise_model_fez,
            "coupling_map": list(coupling_map_fez.get_edges()),
            "basis_gates": basis_gates_fez,
        },
        "run_options": {"shots": 512, "seed": None},
    }
)

history_fez = {"iter": 0, "energies": []}

def cost_function_fez(params):
    pub = (ansatz_fez, [qubit_ham_fez], [params])
    result = noisy_estimator_fez.run(pubs=[pub]).result()
    energy = float(result[0].data.evs[0])
    history_fez["iter"] += 1
    history_fez["energies"].append(energy)
    if history_fez["iter"] % 5 == 0:
        print(f"  iter {history_fez['iter']:3d}: E = {energy:.6f} Ha")
    return energy

# random init to avoid theta=0 local minimum
rng = np.random.default_rng(42)
x0 = rng.uniform(-np.pi, np.pi, ansatz_fez.num_parameters)


result_fez = spsa.minimize(cost_function_fez, x0=x0)

# correct total energy
total_fez = (
    result_fez.fun
    + problem.nuclear_repulsion_energy
    # + problem.hamiltonian.constants["frozen_core"]
)
print(f"\nFakeFez noisy VQE : {total_fez:.6f} Ha")

In [ ]:
# check FakeFez actual noise parameters (q2, q3, q4, q16 only)
props = fake_backend.properties()

print("Single-qubit gate errors (q2, q3, q4, q16):")
for gate in ["sx", "x"]:
    for qubit in [2, 3, 4, 16]:
        try:
            err = props.gate_error(gate, [qubit])
            print(f"  {gate} q{qubit}: {err:.2e}")
        except:
            pass

print("\nTwo-qubit gate errors:")
for pair in [(2,3),(3,4),(3,16)]:
    try:
        err = props.gate_error("cz", list(pair))
        print(f"  cz {pair}: {err:.2e}")
    except Exception as e:
        print(f"  cz {pair}: {e}")

print("\nT1, T2 (q2, q3, q4, q16):")
for qubit in [2, 3, 4, 16]:
    try:
        t1 = props.t1(qubit)
        t2 = props.t2(qubit)
        print(f"  q{qubit}: T1={t1*1e6:.1f}μs, T2={t2*1e6:.1f}μs")
    except:
        pass

# Build coupling map and noise model for topology of real device 

In [ ]:
from qiskit.transpiler import CouplingMap
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error
from qiskit_aer import AerSimulator
from qiskit import transpile
import qiskit_aer.noise as noise

# Noise model for cross 5q topology 
p1 = 2.2e-4   # single-qubit depolarizing error
p2 = 2e-3     # two-qubit depolarizing error
t1 = 15e-6   # T1 relaxation time (seconds)
t2 = 10e-6   # T2 dephasing time (seconds)

# gate times (seconds) - adjust based on actual hardware calibration
gate_time_1q = 50e-9    # 50 ns for rx, ry, rz
gate_time_2q = 200e-9   # 200 ns for cz

# depolarizing errors
err1_dep = noise.depolarizing_error(p1, 1)
err2_dep = noise.depolarizing_error(p2, 2)

# thermal relaxation errors
err1_thermal = thermal_relaxation_error(t1, t2, gate_time_1q)
err2_thermal = thermal_relaxation_error(t1, t2, gate_time_2q).expand(
    thermal_relaxation_error(t1, t2, gate_time_2q)
)

# combine depolarizing + thermal relaxation
err1_combined = err1_dep.compose(err1_thermal)
err2_combined = err2_dep.compose(err2_thermal)

noise_model = noise.NoiseModel()
noise_model.add_all_qubit_quantum_error(err1_combined, ["rx", "ry", "rz"])
noise_model.add_all_qubit_quantum_error(err2_combined, ["cz"])

# Build simulator
cross_sim = AerSimulator(
    noise_model=noise_model,
    coupling_map=cross_5q,
    basis_gates=["rz", "rx", "ry", "cz"],
)

# Transpile ansatz to cross 5q topology
initial_layout={
    ansatz.qregs[0][0]: 1,
    ansatz.qregs[0][1]: 3,
    ansatz.qregs[0][2]: 2,  # center
    ansatz.qregs[0][3]: 4,
}
ansatz_cross = transpile(
    ansatz,
    coupling_map=cross_5q,
    basis_gates=["rz", "rx", "ry", "cz"],
    optimization_level=3,
    seed_transpiler=42,
    initial_layout=initial_layout,
)

print(f"Depth : {ansatz_cross.depth()}")
print(f"CZ    : {ansatz_cross.count_ops().get('cz', 0)}")
print(f"Layout: {ansatz_cross.layout.final_index_layout()}")

In [ ]:
ansatz_cross.draw("mpl", style = "clifford")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_edges_from(cross_5q.get_edges())

pos = {0: (-1, 0), 1: (0, 1), 2: (0, 0), 3: (0, -1), 4: (1, 0)}

# LiH uses q0, q1, q2, q3 (4 qubits), q4 unused
colors = []
for n in G.nodes():
    if n == 2:
        colors.append("#D62728")   # center q2 (most connected)
    elif n in [1, 3, 4]:
        colors.append("#FF7F0E")   # arm qubits used by LiH
    else:
        colors.append("#888888")   # q0 unused

# highlight edges used by LiH
edge_colors = []
for e in G.edges():
    if set(e) in [{1,2}, {3,2}, {4,2}]:
        edge_colors.append("#D62728")
    else:
        edge_colors.append("#CCCCCC")

nx.draw(G, pos=pos, with_labels=True,
        node_color=colors,
        edge_color=edge_colors,
        node_size=800, font_color="white", font_size=14,
        width=2)

plt.title("Cross 5q topology — LiH uses (q0 unused)")
plt.show()

In [ ]:
## calculate fixed distance vqe of HEA with spsa using defined cross_5q 
qubit_ham_cross = qubit_hamiltonian.apply_layout(ansatz_cross.layout)

noisy_estimator_cross = AerEstimator(
    options={
        "backend_options": {
            "noise_model": noise_model,
            "coupling_map": list(cross_5q.get_edges()),
            "basis_gates": ["rz", "ry", "rx", "cz"],
        },
        "run_options": {"shots": 512, "seed": None},
    }
)

history_cross = {"iter": 0, "energies": []}

def cost_function_cross(params):
    pub = (ansatz_cross, [qubit_ham_cross], [params])
    result = noisy_estimator_cross.run(pubs=[pub]).result()
    energy = float(result[0].data.evs[0])
    history_cross["iter"] += 1
    history_cross["energies"].append(energy)
    if history_cross["iter"] % 5 == 0:
        print(f"  iter {history_cross['iter']:3d}: E = {energy:.6f} Ha")
    return energy


x0 = rng.uniform(-np.pi, np.pi, ansatz_cross.num_parameters)


result_cross = spsa.minimize(cost_function_cross, x0=x0)

total_cross = (
    result_cross.fun
    + problem.nuclear_repulsion_energy
)
print(f"\nCross topology noisy VQE : {total_cross:.6f} Ha")
print(f"FCI total                : {fci_electronic + problem.nuclear_repulsion_energy:.6f} Ha")
print(f"Error                    : {(result_cross.fun - fci_electronic)*1000:+.4f} mHa")

In [ ]:
def run_vqe_noisy_at_distance(R, backend_type="fakefez", optimizer="spsa", n_runs=1):
    """
    backend_type: "fakefez" or "cross"
    optimizer: "spsa" or "lbfgsb"
    n_runs: number of independent runs to average (paper uses 100)
    Note: lbfgsb not recommended for noisy environment (shot noise corrupts gradients)
    """
    drv = PySCFDriver(
        atom=f"Li 0.0 0.0 0.0; H 0.0 0.0 {R}",
        basis="sto3g", charge=0, spin=0, unit=DistanceUnit.ANGSTROM,
    )

    prob_ref = drv.run()
    frozen_shift = ActiveSpaceTransformer(2, 3).transform(prob_ref).hamiltonian.constants["ActiveSpaceTransformer"]

    prob = drv.run()
    prob = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3).transform(prob)
    prob.hamiltonian.constants["frozen_core"] = frozen_shift

    mp = ParityMapper(num_particles=prob.num_particles)
    H = mp.map(prob.hamiltonian.second_q_op())

    shift = prob.nuclear_repulsion_energy
    fci_elec = float(np.linalg.eigvalsh(H.to_matrix())[0])

    ans = efficient_su2(
        num_qubits=4, reps=1,
        entanglement=[[0,2],[1,2],[3,2]],
        su2_gates=["rz", "ry"],
    )

    if backend_type == "fakefez":
        ans_t = transpile(
            ans,
            backend=fake_backend,
            optimization_level=3,
            initial_layout={
                ans.qregs[0][0]: 2,
                ans.qregs[0][1]: 4,
                ans.qregs[0][2]: 3,
                ans.qregs[0][3]: 16,
            },
            seed_transpiler=42,
        )
        H_t = H.apply_layout(ans_t.layout)
        est = AerEstimator(options={
            "backend_options": {
                "noise_model": noise_model_fez,
                "coupling_map": list(coupling_map_fez.get_edges()),
                "basis_gates": basis_gates_fez,
            },
            "run_options": {"shots": 512, "seed": None},
        })

    else:  # cross
        ans_t = transpile(
            ans,
            coupling_map=cross_5q,
            basis_gates=["rz", "rx", "ry", "cz"],
            optimization_level=3,
            seed_transpiler=42,
            initial_layout={
                ans.qregs[0][0]: 1,
                ans.qregs[0][1]: 3,
                ans.qregs[0][2]: 2,
                ans.qregs[0][3]: 4,
            },
        )
        H_t = H.apply_layout(ans_t.layout)
        est = AerEstimator(options={
            "backend_options": {
                "noise_model": noise_model,
                "coupling_map": list(cross_5q.get_edges()),
                "basis_gates": ["rz", "rx", "ry", "cz"],
            },
            "run_options": {"shots": 512, "seed": None},
        })

    def cost(p):
        pub = (ans_t, [H_t], [p])
        return float(est.run(pubs=[pub]).result()[0].data.evs[0])

    initial_elec = cost(np.zeros(ans_t.num_parameters))

    all_energies = []
    for i in range(n_runs):
        rng = np.random.default_rng(i)
        if optimizer == "spsa":
            x0 = rng.uniform(-np.pi, np.pi, ans_t.num_parameters)
            res = spsa.minimize(cost, x0=x0)
            all_energies.append(float(res.fun))
        else:
            best_energy = np.inf
            for _ in range(3):
                x0 = rng.uniform(-np.pi, np.pi, ans_t.num_parameters)
                res = minimize(cost, x0=x0, method="L-BFGS-B",
                              options={"maxiter": 200, "ftol": 1e-12, "gtol": 1e-8})
                if res.fun < best_energy:
                    best_energy = res.fun
            all_energies.append(best_energy)

    vqe_elec = np.mean(all_energies)

    return initial_elec + shift, vqe_elec + shift, fci_elec + shift, n_runs

In [ ]:
# Noisy VQE PEC scan: FakeFez vs Cross 5q, SPSA vs L-BFGS-B (LiH)
E_fez_spsa, E_fez_lbfgs, E_cross_spsa, E_cross_lbfgs, E_fci_n = [], [], [], [], []
n_runs = 10
for R in distances:
    _, vqe_fez_spsa,    fci_f, n_spsa   = run_vqe_noisy_at_distance(float(R), backend_type="fakefez", optimizer="spsa", n_runs=n_runs)
    _, vqe_fez_lbfgs,   _, _            = run_vqe_noisy_at_distance(float(R), backend_type="fakefez", optimizer="lbfgsb")
    _, vqe_cross_spsa,  _, _            = run_vqe_noisy_at_distance(float(R), backend_type="cross",   optimizer="spsa", n_runs=n_runs)
    _, vqe_cross_lbfgs, _ , _           = run_vqe_noisy_at_distance(float(R), backend_type="cross",   optimizer="lbfgsb")
    E_fez_spsa.append(vqe_fez_spsa)
    E_fez_lbfgs.append(vqe_fez_lbfgs)
    E_cross_spsa.append(vqe_cross_spsa)
    E_cross_lbfgs.append(vqe_cross_lbfgs)
    E_fci_n.append(fci_f)
    print(f"R={R:.2f}  FakeFez(SPSA)={vqe_fez_spsa:.4f}  FakeFez(L-BFGS-B)={vqe_fez_lbfgs:.4f}  Cross(SPSA)={vqe_cross_spsa:.4f}  Cross(L-BFGS-B)={vqe_cross_lbfgs:.4f}")

E_fez_spsa, E_fez_lbfgs, E_cross_spsa, E_cross_lbfgs, E_fci_n = map(
    np.array, (E_fez_spsa, E_fez_lbfgs, E_cross_spsa, E_cross_lbfgs, E_fci_n)
)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1.2]})

ax1.plot(distances, E_fez_spsa,    "o--", color="#1f77b4", lw=1.6, ms=4, label=f"FakeFez (SPSA, n={n_spsa})")
ax1.plot(distances, E_fez_lbfgs,   "o-",  color="#aec7e8", lw=1.6, ms=4, label="FakeFez (L-BFGS-B)")
ax1.plot(distances, E_cross_spsa,  "s--", color="#d62728", lw=1.6, ms=4, label=f"Cross 5q (SPSA, n={n_spsa})")
ax1.plot(distances, E_cross_lbfgs, "s-",  color="#f7a9a8", lw=1.6, ms=4, label="Cross 5q (L-BFGS-B)")
ax1.plot(distances, E_fci_n,       ":",   color="#2ca02c", lw=2.0, label="FCI")
ax1.set_ylabel("Energy (Hartree)")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(alpha=0.3)
ax1.set_title("LiH dissociation curve — Noisy HEA-VQE: FakeFez vs Cross 5q, SPSA vs L-BFGS-B")

ax2.plot(distances, E_fez_spsa   - E_fci_n, "o--", color="#1f77b4", lw=1.4, ms=4, label=f"FakeFez (SPSA, n={n_spsa})")
ax2.plot(distances, E_fez_lbfgs  - E_fci_n, "o-",  color="#aec7e8", lw=1.4, ms=4, label="FakeFez (L-BFGS-B)")
ax2.plot(distances, E_cross_spsa - E_fci_n, "s--", color="#d62728", lw=1.4, ms=4, label=f"Cross 5q (SPSA, n={n_spsa})")
ax2.plot(distances, E_cross_lbfgs- E_fci_n, "s-",  color="#f7a9a8", lw=1.4, ms=4, label="Cross 5q (L-BFGS-B)")
ax2.axhline(0, color="k", lw=0.5)
ax2.axhline( 1.6e-3, color="gray", ls=":", lw=0.8)
ax2.axhline(-1.6e-3, color="gray", ls=":", lw=0.8)
ax2.set_xlabel("Li-H distance (Å)")
ax2.set_ylabel("Error vs FCI (Ha)")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()